# Notebook 3: Modeling — KMeans Clustering, Evaluation & Recommendation
---

### Steps:
1. Load cleaned training data and define feature matrix
2. Find optimal K using Elbow Method (KneeLocator) and Silhouette Score
3. Train final KMeans model
4. Analyze and describe clusters in human terms
5. Apply model to `recommend.csv` and generate recommendations
6. Validate model using `train.csv`
7. Save all outputs

---
## Setup: Imports and Data Loading

Import all required libraries and load the cleaned training data from Notebook 2. We also install `kneed` directly into the active kernel to avoid the ModuleNotFoundError regardless of which environment is running.

In [ ]:
import subprocess
import sys

# Install kneed into the active notebook kernel
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'kneed', '-q'])

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
from kneed import KneeLocator

%matplotlib inline
sns.set_style('whitegrid')

# File paths
BASE_PATH          = '/Users/saadult/Music Rec Algo/Music-Recommendation-Algorithm/data/'
TRAIN_CLEANED_PATH = BASE_PATH + 'train_cleaned.csv'
TRAIN_RAW_PATH     = BASE_PATH + 'train.csv'
RECOMMEND_PATH     = BASE_PATH + 'recommend.csv'
SCALER_PATH        = BASE_PATH + 'scaler.pkl'
INDEX_PATH         = BASE_PATH + 'cleaned_index.pkl'

# Load cleaned training data
df = pd.read_csv(TRAIN_CLEANED_PATH)
print(f'Cleaned training data shape: {df.shape}')
print(f'\nGenre counts in cleaned data:')
print(df['genre'].value_counts())
df.head()

---
## Load Raw Data and Align to Cleaned Index

We load `train.csv` now and immediately align it to the cleaned dataframe's index using the saved `cleaned_index.pkl`. This is done once at the top so `df_raw_filtered` is available throughout the notebook wherever we need artist names, track names, or genre labels.

This is the fix for the `ValueError: Length of values does not match length of index` — we never try to assign cluster labels to the full 28k row dataframe, only to the aligned filtered version.

In [ ]:
# Load raw train.csv for artist/track name display
df_raw = pd.read_csv(TRAIN_RAW_PATH)

# Load the saved index from Notebook 2
# This tells us exactly which rows survived outlier removal
with open(INDEX_PATH, 'rb') as f:
    cleaned_index = pickle.load(f)

# Align df_raw to only the rows that survived cleaning
df_raw_filtered = df_raw.loc[cleaned_index].copy()

print(f'Raw data rows:          {len(df_raw):,}')
print(f'Cleaned data rows:      {len(df):,}')
print(f'Aligned filtered rows:  {len(df_raw_filtered):,}')
print(f'Length match:           {len(df_raw_filtered) == len(df)}')
print(f'\nHip hop songs preserved: {len(df_raw_filtered[df_raw_filtered["genre"] == "hip hop"])}')

---
## Step 1: Define Feature Matrix

We separate the label columns (`genre`, `topic`, etc.) from the scaled score columns. Only the scaled score columns are fed into KMeans — the label columns are retained so we can interpret cluster results in human terms after training.

In [ ]:
# Separate label columns from score columns
# Label columns are used for interpretation only — NOT fed into KMeans
label_cols = ['release_date', 'genre', 'topic', 'age', 'len']
label_cols = [c for c in label_cols if c in df.columns]

score_cols = [c for c in df.columns if c not in label_cols]

# X is the feature matrix KMeans will train on
X = df[score_cols].values

print(f'Label columns (for interpretation): {label_cols}')
print(f'Score columns (fed into KMeans):    {score_cols}')
print(f'Feature matrix X shape: {X.shape} — (songs x features)')

---
## Step 2: Find the Optimal Number of Clusters (K)

We use two complementary methods to find the best K:

**Elbow Method:** Plots inertia vs K and uses KneeLocator to mathematically find the bend — the point where adding more clusters stops being worth it.

**Silhouette Score:** Measures how well-separated the clusters actually are. Higher = better. We pick the K with the highest score.

If both methods agree → strong choice. If they disagree → we explain our reasoning.

In [ ]:
# Test K = 2 through 12
# This cell may take 1-2 minutes to run
K_RANGE = range(2, 13)
inertias = []
silhouette_scores = []

print('Testing K values...')
for k in K_RANGE:
    km = KMeans(
        n_clusters=k,
        init='k-means++',  # smarter initialization than random
        n_init=10,         # run 10 times, keep best result
        random_state=42,   # reproducibility
        max_iter=300
    )
    km.fit(X)
    inertias.append(km.inertia_)

    sil = silhouette_score(
        X, km.labels_,
        sample_size=5000,  # sample for speed
        random_state=42
    )
    silhouette_scores.append(sil)
    print(f'  K={k:2d} | inertia={km.inertia_:,.1f} | silhouette={sil:.4f}')

print('Done!')

### 2a. Elbow Method with KneeLocator

KneeLocator mathematically identifies the sharpest bend in the inertia curve — the point of diminishing returns where adding more clusters no longer significantly improves model tightness.

In [ ]:
# KneeLocator detects the elbow bend automatically
kneedle = KneeLocator(
    x=list(K_RANGE),
    y=inertias,
    curve='convex',
    direction='decreasing'
)
elbow_k = kneedle.knee

kneedle.plot_knee()
plt.title('Elbow Method — Knee Detected by KneeLocator', fontweight='bold')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia')
plt.tight_layout()
plt.show()

print(f'KneeLocator detected elbow at K = {elbow_k}')

### 2b. Silhouette Score Plot

Music data typically produces silhouette scores between 0.1 and 0.3 because songs naturally overlap across lyrical themes. A lower score does not indicate a failed model — it reflects the nature of the data. We look for the relative peak.

In [ ]:
# Silhouette score plot — look for the peak
best_sil_k = list(K_RANGE)[silhouette_scores.index(max(silhouette_scores))]

plt.figure(figsize=(10, 5))
plt.plot(list(K_RANGE), silhouette_scores, 'rs-', markersize=7, linewidth=2)
plt.axvline(
    x=best_sil_k,
    color='green',
    linestyle='--',
    label=f'Best K = {best_sil_k} (score = {max(silhouette_scores):.4f})'
)
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Score — Find the Peak', fontweight='bold')
plt.xticks(list(K_RANGE))
plt.legend()
plt.tight_layout()
plt.show()

print(f'Best K by silhouette score: {best_sil_k}')
print(f'Best silhouette score:      {max(silhouette_scores):.4f}')

### 2c. K Decision

We compare both methods and select the final K. If they agree that K is used directly. If they disagree we default to the silhouette score since it measures actual cluster quality.

In [ ]:
# Compare both methods and finalize K
print(f'KneeLocator (Elbow) suggests: K = {elbow_k}')
print(f'Silhouette Score suggests:    K = {best_sil_k}')
print()

if elbow_k == best_sil_k:
    OPTIMAL_K = elbow_k
    print(f'Both methods agree — strong choice: K = {OPTIMAL_K}')
else:
    OPTIMAL_K = best_sil_k
    print(f'Methods disagree — using silhouette recommendation: K = {OPTIMAL_K}')
    print('Silhouette preferred: measures actual cluster quality not just compactness.')

print(f'\nFINAL CHOICE: K = {OPTIMAL_K}')

**Finding:** *(Update this cell with your actual K result and reasoning — Report Question iii)*

---
## Step 3: Train the Final KMeans Model

We train one final model with the chosen K. `n_init=20` tries 20 random starting configurations and keeps the best — more reliable than the default of 10.

In [ ]:
# Train final KMeans model with optimal K
final_model = KMeans(
    n_clusters=OPTIMAL_K,
    init='k-means++',
    n_init=20,
    random_state=42,
    max_iter=300
)

final_model.fit(X)

# Add cluster labels to both dataframes
df['cluster'] = final_model.labels_
df_raw_filtered['cluster'] = final_model.labels_

print(f'Model trained with K = {OPTIMAL_K}')
print(f'Final inertia: {final_model.inertia_:,.2f}')
print(f'\nSongs per cluster:')
print(df['cluster'].value_counts().sort_index())

---
## Step 4: Analyze and Describe the Clusters

Now we interpret what each cluster means. We look at the average score per feature per cluster, what genres landed in each cluster, and sample songs. The goal is to describe each cluster in human terms for Report Question iv.

In [ ]:
# Cluster profile — average score per feature per cluster
cluster_profile = df.groupby('cluster')[score_cols].mean().round(3)
print('Average score per cluster:')
cluster_profile

### 4a. Cluster Profile Heatmap

Each row is one cluster. Each column is one feature. Dark red cells indicate that cluster scores highly on that theme. Scan each row for its darkest cells — those are the defining characteristics of that cluster.

In [ ]:
# Cluster profile heatmap — dark red = cluster scores high on that feature
plt.figure(figsize=(16, max(4, OPTIMAL_K)))
sns.heatmap(
    cluster_profile,
    annot=True,
    fmt='.2f',
    cmap='YlOrRd',
    linewidths=0.5,
    yticklabels=[f'Cluster {i}' for i in range(OPTIMAL_K)]
)
plt.title(
    'Cluster Profiles — Average Score per Feature per Cluster',
    fontweight='bold',
    fontsize=13
)
plt.tight_layout()
plt.show()

**Finding:** *(Update after running — describe each cluster in 1 sentence. Report Question iv)*

- **Cluster 0:**
- **Cluster 1:**
- **Cluster 2:**
- **Cluster 3:**
- **Cluster 4:**

### 4b. Genre Breakdown by Cluster

This stacked bar chart shows what proportion of each cluster comes from each genre. With hip hop now preserved in the dataset, we expect to see it appear in whichever cluster captures high obscene content.

In [ ]:
# Genre proportions within each cluster
if 'genre' in df.columns:
    genre_cluster = pd.crosstab(
        df['cluster'],
        df['genre'],
        normalize='index'
    ).round(2)

    genre_cluster.plot(
        kind='bar',
        stacked=True,
        figsize=(14, 6),
        colormap='tab10'
    )
    plt.xlabel('Cluster')
    plt.ylabel('Proportion of Songs')
    plt.title('Genre Composition Within Each Cluster', fontweight='bold')
    plt.legend(title='Genre', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

### 4c. Sample Songs from Each Cluster

We print 5 sample songs from each cluster using `df_raw_filtered` — the version of train.csv aligned to the cleaned dataframe's index. This gives us artist and track names for interpretation while keeping cluster assignments correct.

In [ ]:
# Print 5 sample songs per cluster using the aligned raw dataframe
display_cols = [
    c for c in ['artist_name', 'track_name', 'genre', 'topic', 'cluster']
    if c in df_raw_filtered.columns
]

for cluster_id in sorted(df_raw_filtered['cluster'].unique()):
    print(f'\n--- CLUSTER {cluster_id} ---')
    sample = (
        df_raw_filtered[df_raw_filtered['cluster'] == cluster_id][display_cols]
        .sample(
            min(5, len(df_raw_filtered[df_raw_filtered['cluster'] == cluster_id])),
            random_state=42
        )
    )
    print(sample.to_string(index=False))

---
## Step 5: Save Trained Model and Labeled Training Data

In [ ]:
# Save the trained KMeans model
MODEL_PATH = BASE_PATH + 'kmeans_model.pkl'
with open(MODEL_PATH, 'wb') as f:
    pickle.dump(final_model, f)
print(f'Model saved to: {MODEL_PATH}')

# Save cleaned training data with cluster labels
TRAIN_CLUSTERS_PATH = BASE_PATH + 'train_with_clusters.csv'
df.to_csv(TRAIN_CLUSTERS_PATH, index=False)
print(f'Training data with clusters saved to: {TRAIN_CLUSTERS_PATH}')
print(f'Shape: {df.shape}')

---
## Step 6: Apply Model to Recommend.csv

We load the 11 recommendation test songs, scale them using the saved scaler from Notebook 2, and predict a cluster label for each song. We do not re-fit the scaler — using the training scaler ensures the test data is on the same scale as the model learned from.

In [ ]:
# Load the 11 recommendation test songs
df_rec = pd.read_csv(RECOMMEND_PATH)
print(f'Recommend dataset shape: {df_rec.shape}')
df_rec.head(20)

In [ ]:
# Load the saved scaler from Notebook 2
with open(SCALER_PATH, 'rb') as f:
    scaler = pickle.load(f)
print('Scaler loaded.')

# Add any missing columns as 0 so shapes match
for col in score_cols:
    if col not in df_rec.columns:
        df_rec[col] = 0.0
        print(f'  Added missing column: {col}')

# Scale using the TRAINING scaler — do not re-fit
X_rec = scaler.transform(df_rec[score_cols])
print(f'Recommend feature matrix shape: {X_rec.shape}')

### 6a. Predict Clusters for Recommend Songs

`.predict()` assigns each song to its nearest cluster centroid. The model measures distance from each new song to each of the K centroids and assigns the closest one.

In [ ]:
# Predict cluster for each of the 11 recommendation songs
df_rec['cluster'] = final_model.predict(X_rec)

display_rec_cols = [
    c for c in ['artist_name', 'track_name', 'genre', 'topic', 'cluster']
    if c in df_rec.columns
]
print('Cluster assignments for recommendation songs:')
print(df_rec[display_rec_cols].to_string(index=False))

### 6b. Songs Grouped by Cluster

Songs that landed in the same cluster share the most similar lyrical profiles. This is the basis of the recommendation engine.

In [ ]:
# Group recommendation songs by assigned cluster
print('=== RECOMMENDED SONGS GROUPED BY CLUSTER ===')
print('Songs in the same cluster share the most similar lyrical themes.\n')

for cluster_id in sorted(df_rec['cluster'].unique()):
    songs = df_rec[df_rec['cluster'] == cluster_id]
    print(f'--- Cluster {cluster_id} ({len(songs)} song(s)) ---')
    for _, row in songs.iterrows():
        artist = row.get('artist_name', 'Unknown')
        track  = row.get('track_name', 'Unknown')
        genre  = row.get('genre', 'Unknown')
        print(f'  {artist} - {track} ({genre})')
    print()

### 6c. Find Similar Songs from Training Data

For each recommendation song we find 5 songs from the same cluster in the training data. These are songs with similar lyrical DNA — natural recommendations. This answers Report Question v.

In [ ]:
# Generate recommendations from training data by cluster membership
print('=== SONG RECOMMENDATIONS BASED ON CLUSTER SIMILARITY ===\n')

rec_display_cols = [
    c for c in ['artist_name', 'track_name', 'genre', 'release_date']
    if c in df_raw_filtered.columns
]

for _, rec_song in df_rec.iterrows():
    cluster_id  = rec_song['cluster']
    song_name   = rec_song.get('track_name', 'Unknown')
    artist_name = rec_song.get('artist_name', 'Unknown')

    print(f'Because you like: "{song_name}" by {artist_name} -> Cluster {cluster_id}')
    print('You might also enjoy:')

    cluster_songs = df_raw_filtered[df_raw_filtered['cluster'] == cluster_id]
    similar = cluster_songs[rec_display_cols].sample(
        min(5, len(cluster_songs)),
        random_state=42
    )

    for _, s in similar.iterrows():
        print(
            f"  - {s.get('artist_name', '?')} - "
            f"{s.get('track_name', '?')} "
            f"({s.get('genre', '?')})"
        )
    print()

**Finding:** *(Update after running — Report Question v)*

Note which cluster each recommendation song landed in. Did songs that feel similar end up together? Did any grouping surprise you? Explain any unexpected results.

---
## Step 7: Model Validation

We validate by checking whether the highest-obscene cluster is now dominated by hip hop as predicted in EDA — confirming the per-genre outlier fix worked correctly.

In [ ]:
# Topic distribution across clusters — does each cluster capture a distinct topic?
if 'topic' in df_raw_filtered.columns:
    topic_cluster = pd.crosstab(
        df_raw_filtered['cluster'],
        df_raw_filtered['topic'],
        normalize='index'
    ).round(2)

    plt.figure(figsize=(14, max(4, OPTIMAL_K)))
    sns.heatmap(
        topic_cluster,
        annot=True,
        fmt='.2f',
        cmap='YlOrRd',
        linewidths=0.5
    )
    plt.title(
        'Topic Distribution Within Each Cluster — Validation Check',
        fontweight='bold'
    )
    plt.tight_layout()
    plt.show()

In [ ]:
# Confirm hip hop is now present and check which cluster it landed in
print(f'Hip hop songs in cleaned dataset: {len(df_raw_filtered[df_raw_filtered["genre"] == "hip hop"])}')
print()
print('Hip hop cluster distribution:')
print(
    df_raw_filtered[df_raw_filtered['genre'] == 'hip hop']['cluster']
    .value_counts()
)

# Which cluster has the highest average obscene score?
cluster_means = df.groupby('cluster')[score_cols].mean()
top_obscene_cluster = cluster_means['obscene'].idxmax()

print(f'\nCluster with highest average obscene score: Cluster {top_obscene_cluster}')
print('\nGenre breakdown of that cluster:')
print(
    df_raw_filtered[df_raw_filtered['cluster'] == top_obscene_cluster]['genre']
    .value_counts(normalize=True)
    .round(2)
)
print('\n(Hip hop should now dominate this cluster — confirming the EDA finding.)')

**Validation finding:** *(Update after running.)*

Confirm whether hip hop now appears in the highest-obscene cluster. Compare this result to the previous version where hip hop was completely absent. Note whether the topic distribution heatmap shows meaningful alignment between lyrical topics and cluster assignments.

---
## Step 8: Save All Outputs

In [ ]:
# Save recommendation data with cluster labels
REC_CLUSTERS_PATH = BASE_PATH + 'recommend_with_clusters.csv'
df_rec.to_csv(REC_CLUSTERS_PATH, index=False)
print(f'Recommend data with clusters saved to: {REC_CLUSTERS_PATH}')

print('\n=== All output files ===')
print(f'  {MODEL_PATH}')
print(f'  {TRAIN_CLUSTERS_PATH}')
print(f'  {REC_CLUSTERS_PATH}')

---
## Modeling Summary

| Step | Action | Result |
|---|---|---|
| Index alignment | `df_raw.loc[cleaned_index]` | Resolved length mismatch error |
| Feature matrix | 11 scaled score columns from Notebook 2 | X shape: rows x 11 |
| K selection | Elbow (KneeLocator) + Silhouette Score | K = *[fill in]* |
| Model | KMeans (k-means++, n_init=20) | Inertia = *[fill in]* |
| Hip hop validation | Check top obscene cluster genre breakdown | *[fill in]* |
| Recommendations | Cluster membership from training data | Generated for all 11 songs |

**Report answers covered:**
- **Question iii:** Optimal K → Step 2
- **Question iv:** Cluster descriptions → Step 4
- **Question v:** Recommendations → Step 6